#### This script takes the sector_aggregated EXIOBASE data as inputs and applies a Concordance Matrix to further aggreagate the coundtries to AT and Rest of the world without AT. The outputs as saved under "outputs". As inputs it takes from the "intermediate_outputs" and takes the latest (timestamp) files of F,Y,Z_aggregated

In [1]:
import pandas as pd
import numpy as np
from scipy.linalg import block_diag
import os
from datetime import datetime

In [2]:
# --- Build paths relative to current script directory ---
current_dir = os.getcwd()  # gets current working directory (e.g., python_processing)
base_dir = os.path.abspath(os.path.join(current_dir, '..'))  # one level up → Mini_project
inputs_dir = os.path.join(base_dir, 'inputs')
iot_dir = os.path.join(base_dir, 'inputs', 'EXIOBASE_IOT_2019_ixi')
output_dir = os.path.join(base_dir, 'outputs')
intermediate_outputs_dir = os.path.join(base_dir, 'python_processing','intermediate_outputs')
#concordance_path = os.path.join(inputs_dir, 'C_Matrix_Sector_Aggregation.xlsx')
#material_dir = os.path.join(iot_dir, 'material')  # nested under EXIOBASE_IOT_2019_ixi

In [3]:
# This sets the path to the latest versions (time stamp) of Z_aggregated and Y_aggregated or  F_aggregated
import glob

def get_latest_file_with_prefix(folder, prefix):
    # Pattern to match files starting with prefix + anything
    pattern = os.path.join(folder, prefix + '*')
    files = glob.glob(pattern)

    if not files:
        raise FileNotFoundError(f"No files found with prefix '{prefix}' in {folder}")

    # Sort files by modification time, newest last
    files.sort(key=os.path.getmtime)

    # Return the newest file (last in sorted list)
    return files[-1]

# Test Example Output
latest_Z_path = get_latest_file_with_prefix(intermediate_outputs_dir, 'Z_aggregated')
print("Latest Z aggregation file:", latest_Z_path)

Latest Z aggregation file: c:\Users\simsam\Desktop\Mini_Project\python_processing\intermediate_outputs\Z_aggregated_2025-06-05_18-34-11.xlsx


In [4]:
# --- Build paths relative to current script directory ---
current_dir = os.getcwd()  # gets current working directory (e.g., python_processing)
base_dir = os.path.abspath(os.path.join(current_dir, '..'))  # one level up → Mini_project
#iot_dir = os.path.join(base_dir, 'inputs', 'EXIOBASE_IOT_2019_ixi')
inputs_dir = os.path.join(base_dir, 'inputs')
intermediate_outputs_dir = os.path.join(base_dir, 'python_processing','intermediate_outputs')
#concordance_path = os.path.join(inputs_dir, 'C_Matrix_Sector_Aggregation.xlsx')
#material_dir = os.path.join(iot_dir, 'material')  # nested under EXIOBASE_IOT_2019_ixi
material_dir = os.path.join(iot_dir, 'material')  # nested under EXIOBASE_IOT_2019_ixi
emissions_dir = os.path.join(iot_dir, 'air_emissions')  # nested under EXIOBASE_IOT_2019_ixi

# File paths, using the "get_latest_file_with_prefix" to ensure using latest file version
C_Sector_Aggregation_path = os.path.join(intermediate_outputs_dir, 'C_Sector.txt')
Z_path = get_latest_file_with_prefix(intermediate_outputs_dir, 'Z_aggregated')
Y_path = get_latest_file_with_prefix(intermediate_outputs_dir, 'Y_aggregated')
F_material_path = get_latest_file_with_prefix(intermediate_outputs_dir, 'F_material_aggregated')
F_GHG_path = get_latest_file_with_prefix(intermediate_outputs_dir, 'F_GHG_aggregated')
F_material_Y_path = os.path.join(material_dir, 'F_Y.txt')
F_GHG_Y_path = os.path.join(emissions_dir, 'F_Y.txt')

In [5]:
# --- Load dataframes ---
Y_aggregated = pd.read_excel(Y_path, header=[0, 1], index_col=[0, 1])
Z_aggregated = pd.read_excel(Z_path, header=[0, 1], index_col=[0, 1])
F_material_aggregated = pd.read_excel(F_material_path, header=[0, 1], index_col=[0])
F_GHG_aggregated = pd.read_excel(F_GHG_path, header=[0, 1], index_col=[0])
F_GHG_Y = pd.read_csv(F_GHG_Y_path, sep='\t', header=[0, 1], index_col=[0, 1])
F_material_Y = pd.read_csv(F_material_Y_path, sep='\t', header=[0, 1], index_col=[0, 1])

In [6]:
# Just a check on the sector and region strucutre of Y. Because we wanna build on it below.
# Check MultiIndex levels
print("Row index levels:", Y_aggregated.index.names)
print("Column index levels:", Y_aggregated.columns.names)

# Extract country list (adjust 'region' to your actual level name)
country_list = Y_aggregated.index.get_level_values('Aggregated_Sector').unique().tolist()

print("Countries found:", country_list)

Row index levels: ['Region', 'Aggregated_Sector']
Column index levels: ['region', 'category']
Countries found: ['Plant cultivation', 'Plant processing', 'Animal husbandry', 'Animal processing', 'Other food processing', 'Food waste and reuse', 'Mining and forestry', 'Manufacturing and construction', 'Manufacturing fertilisers', 'Trade, transport and utilities', 'Services', 'Waste', 'Water', 'Hotels and Resteraunts']


#### Generate the C_Matrix_AT_ROW_df (ROW = Rest of the World). This is not automated, but specifically tailored for the austrian case. An automation wouldn't be that hard though.

In [7]:
#We generate the AT + Rest of the World Concordance Matrix. We want the form:
#[ ID    0  ]
#[ 0    ID  ]
#[ 0    ID  ]
#[ 0    ID  ]
# ...
# total 49 blocks stacked vertically

# country_list: list of 49 regions (e.g. from Y_aggregated index)
# sector_list: list of 13 aggregated sectors (e.g. from Y_aggregated index)
# Load Y_aggregated (assuming tab-separated txt, adjust if CSV)
Y_aggregated = pd.read_excel(Y_path, header=[0,1], index_col=[0,1])
#from this we get our country and aggregated sector lists
country_list = Y_aggregated.index.get_level_values('Region').unique().tolist()
sector_list = Y_aggregated.index.get_level_values('Aggregated_Sector').unique().tolist()

# Identity matrix size
n_sectors = len(sector_list)  # 13

ID = np.eye(n_sectors)

# Build first row block: [ID | zeros]
first_row_block = np.hstack([ID, np.zeros((n_sectors, n_sectors))])  # shape (13, 26)

# Build other rows block: [zeros | ID]
other_row_block = np.hstack([np.zeros((n_sectors, n_sectors)), ID])  # shape (13, 26)

# Stack vertically: first one + 48 others
blocks = [first_row_block] + [other_row_block] * (len(country_list) - 1)
matrix_np = np.vstack(blocks)  # shape (49*13, 26)

# Row MultiIndex: all 49 regions × sectors
row_index = pd.MultiIndex.from_product(
    [country_list, sector_list],
    names=['Region', 'Aggregated_Sector']
)

# Column MultiIndex: two "super-regions" — country_A and World_but_country_A — both with same sectors
col_index = pd.MultiIndex.from_product(
    [['AT', 'World_but_AT'], sector_list],
    names=['SuperRegion', 'Aggregated_Sector']
)

# Create DataFrame
C_Matrix_AT_ROW_df = pd.DataFrame(matrix_np, index=row_index, columns=col_index)


In [8]:
# Save C_Matrix_AT_ROW as excel to check
# Get current time as string, e.g. '20250526_153045'
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Build filename with timestamp
filename = f'C_Matrix_AT_ROW_{timestamp}.xlsx'
# Full output path
output_path = os.path.join(os.getcwd(), 'intermediate_outputs', filename)
# Save
C_Matrix_AT_ROW_df.to_excel(output_path)
print(f"Saved to: {output_path}")

Saved to: c:\Users\simsam\Desktop\Mini_Project\python_processing\intermediate_outputs\C_Matrix_AT_ROW_2025-06-05_18-36-45.xlsx


In [9]:
#We generate the AT + Rest of the World Concordance Matrix - but for the final demand! We want the form:
#[ ID    0  ]
#[ 0    ID  ]
#[ 0    ID  ]
#[ 0    ID  ]
# ...
# total 49 blocks stacked vertically

# country_list: list of 49 regions (e.g. from Y_aggregated index)
# final demand: list of 7 aggregated sectors 
# Load Y_aggregated (assuming tab-separated txt, adjust if CSV)
#from this we get our country and aggregated sector lists
country_list = Y_aggregated.index.get_level_values('Region').unique().tolist()
category_list = [
  "Final consumption expenditure by households",
  "Final consumption expenditure by non-profit organisations serving households (NPISH)",
  "Final consumption expenditure by government",
  "Gross fixed capital formation",
  "Changes in inventories",
  "Changes in valuables",
  "Exports: Total (fob)"
]

# Identity matrix size
n_sectors = 7 # 13

ID = np.eye(n_sectors)

# Build first row block: [ID | zeros]
first_row_block = np.hstack([ID, np.zeros((n_sectors, n_sectors))])  # shape (13, 26)

# Build other rows block: [zeros | ID]
other_row_block = np.hstack([np.zeros((n_sectors, n_sectors)), ID])  # shape (13, 26)

# Stack vertically: first one + 48 others
blocks = [first_row_block] + [other_row_block] * (len(country_list) - 1)
matrix_np = np.vstack(blocks)  # shape (49*13, 26)

# Row MultiIndex: all 49 regions × sectors
row_index = pd.MultiIndex.from_product(
    [country_list, category_list],
    names=['Region', 'Aggregated_Sector']
)

# Column MultiIndex: two "super-regions" — country_A and World_but_country_A — both with same sectors
col_index = pd.MultiIndex.from_product(
    [['AT', 'World_but_AT'], category_list],
    names=['SuperRegion', 'Aggregated_Sector']
)

# Create DataFrame
C_Matrix_Y_Countries_df = pd.DataFrame(matrix_np, index=row_index, columns=col_index)


In [10]:
# Save C_Matrix_Y_Countries as excel to check
# Get current time as string, e.g. '20250526_153045'
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Build filename with timestamp
filename = f'C_Matrix_Y_Countries_{timestamp}.xlsx'
# Full output path
output_path = os.path.join(os.getcwd(), 'intermediate_outputs', filename)
# Save
C_Matrix_Y_Countries_df.to_excel(output_path)
print(f"Saved to: {output_path}")

Saved to: c:\Users\simsam\Desktop\Mini_Project\python_processing\intermediate_outputs\C_Matrix_Y_Countries_2025-06-05_18-37-43.xlsx


In [11]:
# Check the Dimensions, are they fitting?
print("C_Matrix_AT_ROW_df:", C_Matrix_AT_ROW_df.shape)
print("C_Matrix_Y_Countries_df", C_Matrix_Y_Countries_df.shape)
print("C_Matrix_Y_Countries_df Row index levels:", C_Matrix_Y_Countries_df.index.names)
print("C_Matrix_Y_Countries_df Column index levels:", C_Matrix_Y_Countries_df.columns.names)
print("Y_aggregated shape:", Y_aggregated.shape)
print("Y_aggregated Row index levels:", Y_aggregated.index.names)
print("Y_aggregated Column index levels:", Y_aggregated.columns.names)
print("F_GHG_Y shape:", F_GHG_Y.shape)
print("F_GHG_Y Row index levels:", F_GHG_Y.index.names)
print("F_GHG_Y Column index levels:", F_GHG_Y.columns.names)
print("F_material_Y shape:", F_material_Y.shape)
print("Z_aggregated shape:", Z_aggregated.shape)
print("F_material_aggregated shape:", F_material_aggregated.shape)
print("F_GHG_aggregated shape:", F_GHG_aggregated.shape)

C_Matrix_AT_ROW_df: (686, 28)
C_Matrix_Y_Countries_df (343, 14)
C_Matrix_Y_Countries_df Row index levels: ['Region', 'Aggregated_Sector']
C_Matrix_Y_Countries_df Column index levels: ['SuperRegion', 'Aggregated_Sector']
Y_aggregated shape: (686, 343)
Y_aggregated Row index levels: ['Region', 'Aggregated_Sector']
Y_aggregated Column index levels: ['region', 'category']
F_GHG_Y shape: (418, 342)
F_GHG_Y Row index levels: ['stressor', None]
F_GHG_Y Column index levels: ['region', 'category']
F_material_Y shape: (61, 342)
Z_aggregated shape: (686, 686)
F_material_aggregated shape: (61, 686)
F_GHG_aggregated shape: (418, 686)


#### Performing the IO-Calculations and saving under "outputs"

In [12]:
# If C_Matrix_Y_Countries_df.index is a flat index with strings like "region|category"
# split it into a MultiIndex
if not isinstance(C_Matrix_Y_Countries_df.index, pd.MultiIndex):
    C_Matrix_Y_Countries_df.index = C_Matrix_Y_Countries_df.index.str.split('|', expand=True)
    C_Matrix_Y_Countries_df.index.names = F_GHG_Y.columns.names
C_aligned = C_Matrix_Y_Countries_df.reindex(F_GHG_Y.columns)
C_aligned = C_aligned.fillna(0)

In [13]:
# --- Load dataframes --- ALREADY LOADED
#Y_aggregated = pd.read_excel(Y_path, header=[0, 1], index_col=[0, 1])
#Z_aggregated = pd.read_excel(Z_path, header=[0, 1], index_col=[0, 1])
#F_material_aggregated = pd.read_excel(F_path, header=[0, 1], index_col=[0])

# --- Matrix operations ---
Y_AT_ROW = C_Matrix_AT_ROW_df.T @ Y_aggregated @ C_Matrix_Y_Countries_df
Z_AT_ROW = C_Matrix_AT_ROW_df.T @ Z_aggregated @ C_Matrix_AT_ROW_df
F_material_AT_ROW = F_material_aggregated @ C_Matrix_AT_ROW_df
F_GHG_AT_ROW = F_GHG_aggregated @ C_Matrix_AT_ROW_df
F_GHG_Y_AT_ROW = F_GHG_Y @ C_aligned

# --- Save with timestamp ---
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

Y_filename = f"Y_AT_ROW_{timestamp}.xlsx"
Z_filename = f"Z_AT_ROW_{timestamp}.xlsx"
F_material_AT_ROW_filename = f"F_material_AT_ROW_{timestamp}.xlsx"
F_GHG_AT_ROW_filename = f"F_GHG_AT_ROW_{timestamp}.xlsx"
F_GHG_Y_AT_ROW_filename = f"F_GHG_Y_AT_ROW_{timestamp}.xlsx"

Y_AT_ROW.to_excel(os.path.join(output_dir, Y_filename))
Z_AT_ROW.to_excel(os.path.join(output_dir, Z_filename))
F_material_AT_ROW.to_excel(os.path.join(output_dir, F_material_AT_ROW_filename))
F_GHG_AT_ROW.to_excel(os.path.join(output_dir, F_GHG_AT_ROW_filename))
F_GHG_Y_AT_ROW.to_excel(os.path.join(output_dir, F_GHG_Y_AT_ROW_filename))

print("Files saved:")
print(f"- {Y_filename}")
print(f"- {Z_filename}")
print(f"- {F_material_AT_ROW_filename}")
print(f"- {F_GHG_AT_ROW_filename}")
print(f"- {F_GHG_Y_AT_ROW_filename}")

Files saved:
- Y_AT_ROW_2025-06-05_18-38-02.xlsx
- Z_AT_ROW_2025-06-05_18-38-02.xlsx
- F_material_AT_ROW_2025-06-05_18-38-02.xlsx
- F_GHG_AT_ROW_2025-06-05_18-38-02.xlsx
- F_GHG_Y_AT_ROW_2025-06-05_18-38-02.xlsx
